# Парсер (Extract) → Очистка (Transform) → DWH (Load) → Дашборд

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import sqlite3

BASE_URL = 'https://books.toscrape.com/catalogue/'


In [2]:
def get_book_category(book_url):
    """Парсит категорию книги по её URL"""
    response = requests.get(book_url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Находим хлебные крошки (breadcrumb)
    breadcrumb = soup.find('ul', class_='breadcrumb')
    if breadcrumb:
        # Категория находится на 3-м месте (после Home и Books)
        category_links = breadcrumb.find_all('li')
        if len(category_links) >= 3:
            return category_links[2].text.strip()
    return 'Unknown'

def parse_book_details(book_url):
    """Парсит детали одной книги"""
    response = requests.get(book_url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Все характеристики из таблицы
    table_rows = soup.find('table', class_='table').find_all('tr')

    book_data = {}
    for row in table_rows:
        key = row.find('th').text.strip()
        value = row.find('td').text.strip()
        book_data[key] = value

    # Описание книги
    description = soup.find('meta', attrs={'name': 'description'})
    if description:
        book_data['description'] = description['content'].strip()
    else:
        book_data['description'] = ''

    return book_data

def parse_page(url):
    """Парсит одну страницу с книгами"""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    books = []

    for book in soup.find_all('article', class_='product_pod'):
        try:
            # Основные данные
            book_name = book.h3.a['title']
            price = float(book.find('p', class_='price_color').text[2:])  # £51.77 → 51.77
            rating_class = book.find('p', class_='star-rating')['class'][1]

            # Преобразуем рейтинг в число
            rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
            rating = rating_map.get(rating_class, 0)

            book_url = BASE_URL + book.h3.a['href']

            # Получаем категорию книги
            category = get_book_category(book_url)

            # Детали книги
            details = parse_book_details(book_url)

            books.append({
                'upc': details.get('UPC', ''),
                'title': book_name,
                'category': category,
                'price_excl_tax': float(details.get('Price (excl. tax)', '0')[2:]),
                'price_incl_tax': float(details.get('Price (incl. tax)', '0')[2:]),
                'tax': float(details.get('Tax', '0')[2:]),
                'availability': details.get('Availability', ''),
                'rating': rating,
                'reviews': int(details.get('Number of reviews', '0')),
                'description': details.get('description', ''),
                'url': book_url
            })
            print(f"{book_name[:50]}... - {price} £ - рейтинг: {rating} - категория: {category}")

            time.sleep(0.1)

        except Exception as e:
            print(f"Ошибка: {e}")
            continue

    return books

def scrape_books(max_books=500):
    """Собирает книги с сайта"""
    all_books = []
    page = 1

    while len(all_books) < max_books:
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"
        print(f"\n Парсинг страницы {page}: {url}")

        books = parse_page(url)
        if not books:
            print("Больше нет книг для парсинга")
            break

        all_books.extend(books)
        print(f"Найдено книг: {len(books)}. Всего собрано: {len(all_books)}")

        if len(books) < 20:
            print("Это последняя страница")
            break

        page += 1

    return all_books[:max_books]

In [3]:
# Запуск
if __name__ == "__main__":
    books_data = scrape_books(max_books=200)

    df = pd.DataFrame(books_data)
    df.to_csv('books_raw.csv', index=False, encoding='utf-8-sig')

    print(f" Собрано книг: {len(df)}")


 Парсинг страницы 1: https://books.toscrape.com/catalogue/page-1.html
A Light in the Attic... - 1.77 £ - рейтинг: 3 - категория: Poetry
Tipping the Velvet... - 3.74 £ - рейтинг: 1 - категория: Historical Fiction
Soumission... - 0.1 £ - рейтинг: 1 - категория: Fiction
Sharp Objects... - 7.82 £ - рейтинг: 4 - категория: Mystery
Sapiens: A Brief History of Humankind... - 4.23 £ - рейтинг: 5 - категория: History
The Requiem Red... - 2.65 £ - рейтинг: 1 - категория: Young Adult
The Dirty Little Secrets of Getting Your Dream Job... - 3.34 £ - рейтинг: 4 - категория: Business
The Coming Woman: A Novel Based on the Life of the... - 7.93 £ - рейтинг: 3 - категория: Default
The Boys in the Boat: Nine Americans and Their Epi... - 2.6 £ - рейтинг: 4 - категория: Default
The Black Maria... - 2.15 £ - рейтинг: 1 - категория: Poetry
Starving Hearts (Triangular Trade Trilogy, #1)... - 3.99 £ - рейтинг: 2 - категория: Default
Shakespeare's Sonnets... - 0.66 £ - рейтинг: 4 - категория: Poetry
Set Me Fr

In [4]:
print(df['category'].value_counts().head(10))

category
Default           31
Sequential Art    23
Nonfiction        20
Young Adult       10
Fantasy           10
Romance            9
Fiction            9
Poetry             9
Food and Drink     9
Add a comment      7
Name: count, dtype: int64


In [5]:
df.head(5)

,upc,title,category,price_excl_tax,price_incl_tax,tax,availability,rating,reviews,description,url
0,a897fe39b1053632,A Light in the Attic,Poetry,1.77,1.77,0.0,In stock (22 available),3,0,It's hard to imagine a world without A Light i...,https://books.toscrape.com/catalogue/a-light-i...
1,90fa61229261140a,Tipping the Velvet,Historical Fiction,3.74,3.74,0.0,In stock (20 available),1,0,"""Erotic and absorbing...Written with starling ...",https://books.toscrape.com/catalogue/tipping-t...
2,6957f44c3847a760,Soumission,Fiction,0.10,0.10,0.0,In stock (20 available),1,0,"Dans une France assez proche de la nôtre, un h...",https://books.toscrape.com/catalogue/soumissio...
3,e00eb4fd7b871a48,Sharp Objects,Mystery,7.82,7.82,0.0,In stock (20 available),4,0,"WICKED above her hipbone, GIRL across her hear...",https://books.toscrape.com/catalogue/sharp-obj...
4,4165285e1663650f,Sapiens: A Brief History of Humankind,History,4.23,4.23,0.0,In stock (20 available),5,0,From a renowned historian comes a groundbreaki...,https://books.toscrape.com/catalogue/sapiens-a...


In [6]:
# Подключаемся к БД (создаст файл dwh_books.db)
conn = sqlite3.connect('dwh_books.db')
cursor = conn.cursor()

In [7]:
# Удаляем старые таблицы, если есть
cursor.execute("DROP TABLE IF EXISTS book_category")
cursor.execute("DROP TABLE IF EXISTS fact_books")
cursor.execute("DROP TABLE IF EXISTS dim_categories")

In [8]:
cursor.execute('''
CREATE TABLE fact_books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    upc TEXT UNIQUE,
    title TEXT,
    category TEXT,
    price_excl_tax REAL,
    price_incl_tax REAL,
    tax REAL,
    rating INTEGER,
    reviews INTEGER,
    availability TEXT,
    description TEXT,
    url TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

# Создаём таблицу измерений (категории)
cursor.execute('''
CREATE TABLE dim_categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE
)
''')

# Создаём таблицу связей
cursor.execute('''
CREATE TABLE book_category (
    book_id INTEGER,
    category_id INTEGER,
    FOREIGN KEY (book_id) REFERENCES fact_books(book_id),
    FOREIGN KEY (category_id) REFERENCES dim_categories(category_id),
    PRIMARY KEY (book_id, category_id)
)
''')

print("Таблицы DWH созданы")

Таблицы DWH созданы


# Загружаем данные в DWH (ETL Load)

In [9]:
# Загружаем данные из CSV
df = pd.read_csv('books_raw.csv', encoding='utf-8-sig')

In [10]:
conn = sqlite3.connect('dwh_books.db')
cursor = conn.cursor()

In [11]:
categories = df['category'].unique()

for cat in categories:
    cursor.execute('''
    INSERT OR IGNORE INTO dim_categories (category_name)
    VALUES (?)
    ''', (cat,))

conn.commit()
print(f"Загружено {len(categories)} категорий")

 Загружено 36 категорий


In [12]:
# Очистка данных
df = df.drop_duplicates(subset=['upc'])  # Убираем дубликаты
df = df.fillna({'description': ''})      # Заполняем пустые описания

In [13]:
for _, row in df.iterrows():
    cursor.execute('''
    INSERT OR REPLACE INTO fact_books (
        upc, title, category, price_excl_tax, price_incl_tax, tax,
        rating, reviews, availability, description, url
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        row['upc'], row['title'], row['category'],
        row['price_excl_tax'], row['price_incl_tax'], row['tax'],
        row['rating'], row['reviews'], row['availability'],
        row['description'], row['url']
    ))

conn.commit()
print(f"Загружено {len(df)} книг в fact_books")

Загружено 200 книг в fact_books


In [14]:
cursor.execute('''
INSERT OR IGNORE INTO book_category (book_id, category_id)
SELECT f.book_id, c.category_id
FROM fact_books f
JOIN dim_categories c ON f.category = c.category_name
''')

conn.commit()

In [15]:
cursor.execute("SELECT COUNT(*) FROM fact_books")
books_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM dim_categories")
cats_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM book_category")
links_count = cursor.fetchone()[0]

print(f"Книг в fact_books: {books_count}")
print(f" Категорий в dim_categories: {cats_count}")
print(f"Связей в book_category: {links_count}")

conn.close()

Книг в fact_books: 200
 Категорий в dim_categories: 36
Связей в book_category: 200


In [17]:
import plotly.express as px
import plotly.graph_objects as go

In [18]:
if len(df_books) > 0:
    fig2 = px.histogram(df_books,
                        x='price_incl_tax',
                        nbins=25,
                        title='Распределение цен на книги',
                        labels={'price_incl_tax': 'Цена (£)', 'count': 'Количество книг'},
                        color_discrete_sequence=['#005BFF'])
    fig2.add_vline(x=df_books['price_incl_tax'].mean(),
                   line_dash="dash",
                   line_color="red",
                   annotation_text=f"Средняя: {df_books['price_incl_tax'].mean():.2f}£")
    fig2.show()

NameError: name 'df_books' is not defined

book_warehouse/
├── 01_parser.py          # Парсер книг
├── 02_create_dwh.py      # Создание таблиц DWH
├── 03_load_data.py       # Загрузка данных в DWH
├── 04_analysis.sql       # SQL аналитические запросы
├── 05_dashboard.py       # Plotly дашборд
├── requirements.txt      # Зависимости
├── dwh_books.db          # База данных (после запуска)
# └── README.md             # Описание проекта

# Итог

Парсер (Extract)- собирает данные с books.toscrape.com

Трансформация (Transform) - очистка, удаление дубликатов

Загрузка (Load) - данные в DWH (SQLite)

DWH модель "Звезда" -fact_books + dim_categories

Дашборд (Plotly) -  интерактивнын графики